In [6]:
# SMOKE=1 (default) = quick test: 1 run, 1 fold, XGB only.
# The sbatch job sets SMOKE=0 for the full experiment.
import os, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

SMOKE  = os.environ.get("SMOKE", "1") == "1"
N_JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK", 4))   # never -1 on Amarel

DATA_DIR = Path(os.environ.get("DATA_DIR", "/scratch/sm3664/medicare-fraud"))
LABELED  = DATA_DIR / "processed/part_b/labeled/part_b_labeled_2019_2023.parquet"
RES_DIR  = DATA_DIR / "results/part_b"; RES_DIR.mkdir(parents=True, exist_ok=True)
RES_CSV  = RES_DIR / ("nb05_smoke.csv" if SMOKE else "nb05_fold_results.csv")
print("SMOKE:", SMOKE, "| N_JOBS:", N_JOBS, "| results →", RES_CSV.name)

SMOKE: True | N_JOBS: 4 | results → nb05_smoke.csv


## Settings

- `N_RUNS = 6`, `N_FOLDS = 5`, as in the paper.
- Hyperparameters are the paper's. XGBoost and RF otherwise use library defaults.
- ⚠️ `USE_ENT_CD`: the paper's features don't include entity type (individual vs. organization). Default `False` replicates the paper. `True` is a possible improvement to test later.

In [7]:
N_RUNS, N_FOLDS = (1, 5) if SMOKE else (6, 5)
FOLDS_TO_RUN    = [0] if SMOKE else list(range(N_FOLDS))
LEARNERS        = ["XGB"] if SMOKE else ["XGB", "RF"]
DATASETS        = ["SbP", "A", "AE"]
USE_ENT_CD      = False

def make_model(name, seed):
    if name == "XGB":
        return XGBClassifier(max_depth=4, n_jobs=N_JOBS, random_state=seed,
                             tree_method="hist", eval_metric="logloss")
    return RandomForestClassifier(max_depth=16, n_jobs=N_JOBS, random_state=seed)

## Step 1 — Load data and sort columns into feature groups

- **Keys** (never features): NPI, year, fraud.
- **Aggregated (30):** the 5 claims fields × min/max/median/mean/sum/std from Notebook 02.
- **SbP (56):** the by-Provider features added in Notebook 03.
- **Common:** provider type, one-hot encoded as in the paper. Used by all three datasets.

The asserts stop the notebook if the column counts don't match what Notebooks 02–03 produced.

In [8]:
df = pd.read_parquet(LABELED)
print(df.shape, "| fraud rows:", int(df.fraud.sum()))

KEYS  = ["Rndrng_NPI", "year", "fraud"]
CATS  = ["Rndrng_Prvdr_Type", "Rndrng_Prvdr_Ent_Cd"]
BASES = ["Tot_Benes", "Tot_Srvcs", "Tot_Bene_Day_Srvcs", "Avg_Sbmtd_Chrg", "Avg_Mdcr_Pymt_Amt"]
STATS = ["min", "max", "median", "mean", "sum", "std"]

AGG = [c for c in df.columns
       if any(c.startswith(b + "_") for b in BASES) and c.rsplit("_", 1)[-1] in STATS]
SBP = [c for c in df.columns if c not in KEYS + CATS + AGG]
print("AGG:", len(AGG), "| SbP:", len(SBP))
assert len(AGG) == 30, AGG
assert len(SBP) == 56, SBP

(5626423, 91) | fraud rows: 2652
AGG: 30 | SbP: 56


In [9]:
cat_df = pd.get_dummies(df["Rndrng_Prvdr_Type"], prefix="ptype", dtype=np.float32)
if USE_ENT_CD:
    cat_df["ent_org"] = (df["Rndrng_Prvdr_Ent_Cd"] == "O").astype(np.float32)
COMMON = list(cat_df.columns)

FEATS = AGG + SBP + COMMON
X_all = np.hstack([df[AGG + SBP].to_numpy(np.float32), cat_df.to_numpy(np.float32)])
y_all = df["fraud"].to_numpy(np.int8)
del cat_df

col_idx = {f: i for i, f in enumerate(FEATS)}
DS_COLS = {
    "SbP": [col_idx[c] for c in SBP + COMMON],
    "A":   [col_idx[c] for c in AGG + COMMON],
    "AE":  [col_idx[c] for c in AGG + SBP + COMMON],
}
for k, v in DS_COLS.items(): print(k, "features:", len(v))
print("X_all:", X_all.shape, f"{X_all.nbytes/1e9:.1f} GB")

SbP features: 160
A features: 134
AE features: 190
X_all: (5626423, 190) 4.3 GB


In [10]:
npi = df["Rndrng_NPI"].to_numpy()
npi_ever_fraud = df.groupby("Rndrng_NPI")["fraud"].max()

# def assign_folds(seed):
#     rng = np.random.default_rng(seed)
#     fold_of = {}
#     for cls in (0, 1):
#         ids = npi_ever_fraud.index[npi_ever_fraud == cls].to_numpy()
#         rng.shuffle(ids)
#         fold_of.update(zip(ids, np.arange(len(ids)) % N_FOLDS))
#     return pd.Series(npi).map(fold_of).to_numpy()
def assign_folds(seed):
    rng = np.random.default_rng(seed)
    parts = []
    for cls in (0, 1):
        ids = rng.permutation(npi_ever_fraud.index[npi_ever_fraud == cls].to_numpy())
        parts.append(pd.Series(np.arange(len(ids)) % N_FOLDS, index=ids))
    fold_of = pd.concat(parts)
    return pd.Series(npi).map(fold_of).to_numpy()
f0 = assign_folds(0)
chk = pd.DataFrame({"npi": npi, "fold": f0, "y": y_all})
assert chk.groupby("npi").fold.nunique().max() == 1, "NPI leaks across folds"
print(chk.groupby("fold").agg(rows=("y", "size"), fraud_rows=("y", "sum"),
                              fraud_npis=("npi", lambda s: s[chk.loc[s.index, "y"] == 1].nunique())))
del chk

         rows  fraud_rows  fraud_npis
fold                                 
0     1126090         549         186
1     1123638         515         186
2     1124656         546         186
3     1125412         523         186
4     1126627         519         186


In [11]:
def evaluate(y_true, p, thr):
    pred = (p >= thr).astype(np.int8)
    tp = ((pred == 1) & (y_true == 1)).sum(); fn = ((pred == 0) & (y_true == 1)).sum()
    tn = ((pred == 0) & (y_true == 0)).sum(); fp = ((pred == 1) & (y_true == 0)).sum()
    tpr, tnr = tp / (tp + fn), tn / (tn + fp)
    return {"AUC": roc_auc_score(y_true, p), "AUPRC": average_precision_score(y_true, p),
            "TPR": tpr, "TNR": tnr, "GMean": np.sqrt(tpr * tnr), "threshold": thr}

In [12]:
done = set()
if RES_CSV.exists():
    prev = pd.read_csv(RES_CSV)
    done = set(zip(prev.run, prev.fold, prev.dataset, prev.learner))
print("Already done:", len(done))

for run in range(N_RUNS):
    folds = assign_folds(run)
    for k in FOLDS_TO_RUN:
        te = np.where(folds == k)[0]; tr = np.where(folds != k)[0]
        y_tr, y_te = y_all[tr], y_all[te]
        thr = y_tr.mean()                          # prior probability threshold
        for ds in DATASETS:
            cols = DS_COLS[ds]
            X_tr = X_all[np.ix_(tr, cols)]; X_te = X_all[np.ix_(te, cols)]
            for lr in LEARNERS:
                if (run, k, ds, lr) in done: continue
                t0 = time.time()
                m = make_model(lr, seed=run).fit(X_tr, y_tr)
                p = m.predict_proba(X_te)[:, 1]
                row = {"run": run, "fold": k, "dataset": ds, "learner": lr,
                       **evaluate(y_te, p, thr), "fit_sec": round(time.time() - t0, 1),
                       "test_pos": int(y_te.sum())}
                pd.DataFrame([row]).to_csv(RES_CSV, mode="a", header=not RES_CSV.exists(), index=False)
                print(f"run{run} fold{k} {ds:>3} {lr}: AUC={row['AUC']:.4f} "
                      f"G={row['GMean']:.4f} ({row['fit_sec']}s)")
            del X_tr, X_te

Already done: 0
run0 fold0 SbP XGB: AUC=0.7502 G=0.6749 (52.3s)
run0 fold0   A XGB: AUC=0.7121 G=0.6285 (46.1s)
run0 fold0  AE XGB: AUC=0.7312 G=0.6513 (70.2s)


In [13]:
res = pd.read_csv(RES_CSV)
print("Results per combo:", res.groupby(["dataset", "learner"]).size().to_dict())

def ci(s):
    m, h = s.mean(), 1.96 * s.std(ddof=1) / np.sqrt(len(s)) if len(s) > 1 else np.nan
    return f"{m:.4f} ({m-h:.4f}, {m+h:.4f})"

t9 = res.pivot_table(index="dataset", columns="learner", values="AUC", aggfunc=ci).reindex(DATASETS)
print("\nTable 9 — AUC mean (95% CI)\n", t9)

x = res[res.learner == "XGB"].groupby("dataset")[["TPR", "TNR", "GMean", "AUPRC"]].mean()
if {"A", "AE"} <= set(x.index):
    t10 = x.loc["AE"].to_frame("AE").assign(lift_vs_A_pct=100 * (x.loc["AE"] / x.loc["A"] - 1))
    print("\nTable 10 — XGB, AE vs A\n", t10.round(4))
t9.to_csv(RES_DIR / ("nb05_table9_smoke.csv" if SMOKE else "nb05_table9.csv"))

Results per combo: {('A', 'XGB'): 1, ('AE', 'XGB'): 1, ('SbP', 'XGB'): 1}

Table 9 — AUC mean (95% CI)
 learner                XGB
dataset                   
SbP      0.7502 (nan, nan)
A        0.7121 (nan, nan)
AE       0.7312 (nan, nan)

Table 10 — XGB, AE vs A
            AE  lift_vs_A_pct
TPR    0.5209         3.2491
TNR    0.8141         3.9811
GMean  0.6513         3.6144
AUPRC  0.0031        95.6992


In [14]:
# Diagnostic (paper Fig. 2): row-level k-fold vs k-fold-by-NPI, AE + XGB, fold 0.
# Also train AUC, to tell underfitting from failure to generalize.
cols = DS_COLS["AE"]
row_folds = np.random.default_rng(0).permutation(len(y_all)) % N_FOLDS

for name, folds in [("row-level", row_folds), ("by-NPI", assign_folds(0))]:
    tr = np.where(folds != 0)[0]; te = np.where(folds == 0)[0]
    m = make_model("XGB", 0).fit(X_all[np.ix_(tr, cols)], y_all[tr])
    auc_te = roc_auc_score(y_all[te], m.predict_proba(X_all[np.ix_(te, cols)])[:, 1])
    auc_tr = roc_auc_score(y_all[tr], m.predict_proba(X_all[np.ix_(tr, cols)])[:, 1])
    print(f"{name:>9}: train AUC={auc_tr:.4f} | test AUC={auc_te:.4f}")

row-level: train AUC=0.9279 | test AUC=0.8348
   by-NPI: train AUC=0.9354 | test AUC=0.7312
